# Deep Learning QuckStart
- MLflow3를 사용하여 Pytorch based인 Iris classifier와 함께 딥러닝 모델 평가 및 추적

In [12]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [13]:
import mlflow
import mlflow.pytorch
from mlflow.entities import Dataset

In [15]:
def prepare_data(df):
    X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
    y = torch.tensor(df.iloc[:, -1].values, dtype=torch.long)
    return X, y

In [16]:
def compute_accuracy(model, X, y):
    with torch.no_grad():
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).sum().item() / y.size(0)
    return accuracy

In [17]:
class IrisClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [18]:
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df["target"] = iris.target

In [19]:
train_df, test_df = train_test_split(iris_df, test_size=0.2, random_state=42)

In [20]:
train_dataset = mlflow.data.from_pandas(train_df, name="train")
X_train, y_train = prepare_data(train_dataset.df)

In [24]:
input_size = X_train.shape[1]
hidden_size = 16
output_size = len(iris.target_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scripted_model = IrisClassifier(input_size, hidden_size, output_size).to(device)
scripted_model = torch.jit.script(scripted_model).to(device)

In [25]:
with mlflow.start_run() as run:
    train_dataset: Dataset = mlflow.data.from_pandas(train_df, name="train")
    X_train, y_train = prepare_data(train_dataset.df)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(scripted_model.parameters(), lr=0.01)

    for epoch in range(101):
        X_train, y_train = X_train.to(device), y_train.to(device)
        out = scripted_model(X_train)
        loss = criterion(out, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            model_info = mlflow.pytorch.log_model(
                pytorch_model=scripted_model,
                name=f"torch-iris-{epoch}",
                step=epoch,
                input_example=X_train.cpu().float().numpy(),
            )

            mlflow.log_params(
                params={
                    "n_layers": 3,
                    "activation": "ReLU",
                    "criterion": "CrossEntropyLoss",
                    "optimizer": "Adam",
                }
            )

            mlflow.log_metric(
                key="accuracy",
                value=compute_accuracy(scripted_model, X_train, y_train),
                step=epoch,
                model_id=model_info.model_id,
                dataset=train_dataset,
            )

2025/07/09 08:11:25 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: RuntimeError('The following operation failed in the TorchScript interpreter.\nTraceback of TorchScript (most recent call last):\nRuntimeError: The following operation failed in the TorchScript interpreter.\nTraceback of TorchScript (most recent call last):\n  File "/tmp/ipykernel_2067203/2701197327.py", line 9, in <forward op>\n    def forward(self, x):\n        x = self.fc1(x)\n            ~~~~~~~~ <--- HERE\n        x = self.relu(x)\n        x = self.fc2(x)\n  File "/home/ubuntu/miniconda3/envs/mlflow/lib/python3.10/site-packages/torch/nn/modules/linear.py", line 125, in forward\n    def forward(self, input: Tensor) -> Tensor:\n        return F.linear(input, self.weight, self.bias)\n               ~~~~~~~~ <--- HERE\nRuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat1

/home/ubuntu/miniconda3/envs/mlflow/lib/python3.10/site-packages/torch/serialization.py:1488: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(
2025/07/09 08:11:29 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      4.599999904632568,
      3.5999999046325684,
      1.0,
      0.20000000298023224
    ],
    [
      5.699999809265137,
      4.400000095367432,
      1.5,
      0.4000000059604645
    ],
    [
      6.699999809265137,
      3.0999999046325684,
      4.400000095367432,
      1.399999976158142
    ],
    [
      4.800000190734863,
      3.4000000953674316,
      1.600000023841858,
      0.20000000298023224
    ],
    [
      4.400000095367432,
      3.200000047683716,
      1.2999999523162842,
      0.20000000298023224
    ],
    [
      6.300000190734863,
      2.5,
      5.0,
      1.899999

🏃 View run nimble-moth-975 at: http://18.181.86.77:5001/#/experiments/0/runs/0e62825ad483401bbb5e4122c1b823be
🧪 View experiment at: http://18.181.86.77:5001/#/experiments/0


In [26]:
ranked_checkpoints = mlflow.search_logged_models(
    filter_string=f"source_run_id='{run.info.run_id}'",
    order_by=[{"field_name": "metrics.accuracy", "ascending": False}],
    output_format="list",
)

In [27]:
best_checkpoint = ranked_checkpoints[0]
print(f"Best model: {best_checkpoint}")
print(best_checkpoint.metrics)

Best model: LoggedModel(artifact_location='mlflow-artifacts:/0/models/m-ed8b517b4ec741268c03687add28ddfd/artifacts', creation_timestamp=1752048725307, experiment_id='0', last_updated_timestamp=1752048729511, model_id='m-ed8b517b4ec741268c03687add28ddfd', model_type='', model_uri='models:/m-ed8b517b4ec741268c03687add28ddfd', name='torch-iris-100', source_run_id='0e62825ad483401bbb5e4122c1b823be', status=<LoggedModelStatus.READY: 'READY'>, status_message='')
[<Metric: dataset_digest='1f1c13b5', dataset_name='train', key='accuracy', model_id='m-ed8b517b4ec741268c03687add28ddfd', run_id='0e62825ad483401bbb5e4122c1b823be', step=100, timestamp=1752048729528, value=0.9833333333333333>]


In [28]:
worst_checkpoint = ranked_checkpoints[-1]
print(f"Worst model: {worst_checkpoint}")
print(worst_checkpoint.metrics)

Worst model: LoggedModel(artifact_location='mlflow-artifacts:/0/models/m-0c61024b9d54473f9b8819ee8c20e446/artifacts', creation_timestamp=1752048685679, experiment_id='0', last_updated_timestamp=1752048689249, model_id='m-0c61024b9d54473f9b8819ee8c20e446', model_type='', model_uri='models:/m-0c61024b9d54473f9b8819ee8c20e446', name='torch-iris-0', source_run_id='0e62825ad483401bbb5e4122c1b823be', status=<LoggedModelStatus.READY: 'READY'>, status_message='')
[<Metric: dataset_digest='1f1c13b5', dataset_name='train', key='accuracy', model_id='m-0c61024b9d54473f9b8819ee8c20e446', run_id='0e62825ad483401bbb5e4122c1b823be', step=0, timestamp=1752048689273, value=0.325>]
